In [ ]:

# ============================================================
# PROJECT 3 - FINANCIAL FRAUD INTELLIGENCE
# REAL ULB DATA | GOOGLE COLAB | SINGLE CELL
# ============================================================

import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "pandas", "numpy", "plotly", "scikit-learn"
])

import pandas as pd
import numpy as np
import plotly.express as px
import zipfile
from pathlib import Path
from plotly.io import to_html
from IPython.display import display

# 1. SETTINGS
PROJECT = Path("/content/Financial_Fraud_Intelligence")
PROJECT.mkdir(parents=True, exist_ok=True)
DOWNLOAD_ZIP = False

DATA_URL = (
    "https://storage.googleapis.com/"
    "download.tensorflow.org/data/creditcard.csv"
)

# 2. DOWNLOAD REAL DATA
print("Loading real credit-card transactions...")

try:
    df = pd.read_csv(DATA_URL, low_memory=False)
    source_used = DATA_URL
    print("Loaded from Google-hosted CSV.")

except Exception as error:
    print("Direct download failed:", error)
    print("Trying OpenML mirror...")

    from sklearn.datasets import fetch_openml

    result = fetch_openml(
        data_id=1597,
        as_frame=True,
        parser="auto"
    )

    df = result.frame.copy()
    source_used = "OpenML dataset 1597"

df.columns = [str(c).strip() for c in df.columns]

required = {"Amount", "Class"}
if not required.issubset(df.columns):
    raise ValueError(
        "Required columns missing: "
        + str(list(df.columns))
    )

print("Original records:", len(df))
print("Columns:", list(df.columns))

# 3. CLEAN DATA
df["Amount"] = pd.to_numeric(
    df["Amount"], errors="coerce"
)
df["Class"] = pd.to_numeric(
    df["Class"], errors="coerce"
)

if "Time" in df.columns:
    df["Time"] = pd.to_numeric(
        df["Time"], errors="coerce"
    )

df = df.dropna(subset=["Amount", "Class"])
df = df[
    (df["Amount"] >= 0)
    & (df["Class"].isin([0, 1]))
].copy()

# Keep genuine repeated observations.
# Do not drop duplicates without evidence they are errors.
df = df.reset_index(drop=True)
df["Class"] = df["Class"].astype(int)

df["Transaction_Type"] = df["Class"].map({
    0: "Legitimate",
    1: "Fraud"
})

df["Amount_Band"] = pd.cut(
    df["Amount"],
    bins=[-0.001, 10, 50, 100, 500, np.inf],
    labels=[
        "0-10", "10-50", "50-100",
        "100-500", "500+"
    ],
    include_lowest=True
)

if "Time" in df.columns:
    df["Elapsed_Hour"] = (
        df["Time"] // 3600
    ).astype("Int64")
    df["Hour_Of_Day"] = (
        (df["Time"] // 3600) % 24
    ).astype("Int64")

if df.empty:
    raise ValueError("No usable transactions.")

df.to_csv(
    PROJECT / "cleaned_fraud_data.csv",
    index=False
)

print("\nCLEANED DATA")
display(df.head())

# 4. KPIs
fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0]

kpis = {
    "Total Transactions": len(df),
    "Fraud Transactions": len(fraud),
    "Legitimate Transactions": len(legit),
    "Fraud Rate (%)": round(
        df["Class"].mean() * 100, 4
    ),
    "Total Transaction Amount": df["Amount"].sum(),
    "Fraud Transaction Amount": fraud["Amount"].sum(),
    "Average Fraud Amount": fraud["Amount"].mean(),
    "Average Legitimate Amount": legit["Amount"].mean()
}

kpi_df = pd.DataFrame(
    list(kpis.items()),
    columns=["KPI", "Value"]
)

print("\nFRAUD KPIs")
display(kpi_df)

kpi_df.to_csv(
    PROJECT / "kpi_summary.csv",
    index=False
)

# 5. INTERACTIVE CHARTS
figures = []

def show_chart(fig, title):
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=500,
        margin=dict(l=45, r=40, t=80, b=55)
    )
    figures.append((title, fig))
    fig.show()

# CHART 1 - TRANSACTION CLASS DISTRIBUTION
d = (
    df.groupby("Transaction_Type")
    .size()
    .reset_index(name="Transactions")
)

show_chart(
    px.bar(
        d,
        x="Transaction_Type",
        y="Transactions",
        color="Transaction_Type",
        log_y=True
    ),
    "1. Fraud vs Legitimate Transactions (Log Scale)"
)

# CHART 2 - FRAUD PERCENTAGE
show_chart(
    px.pie(
        d,
        names="Transaction_Type",
        values="Transactions",
        hole=0.5
    ),
    "2. Transaction Class Share"
)

# CHART 3 - TRANSACTION AMOUNTS
d = (
    df.groupby(
        "Transaction_Type", as_index=False
    )["Amount"].sum()
)

show_chart(
    px.bar(
        d,
        x="Transaction_Type",
        y="Amount",
        color="Transaction_Type"
    ),
    "3. Total Transaction Amount by Class"
)

# CHART 4 - AVERAGE AMOUNT
d = (
    df.groupby(
        "Transaction_Type", as_index=False
    )["Amount"].mean()
)

show_chart(
    px.bar(
        d,
        x="Transaction_Type",
        y="Amount",
        color="Transaction_Type"
    ),
    "4. Average Transaction Amount by Class"
)

# CHART 5 - AMOUNT DISTRIBUTION
sample = df.groupby(
    "Class", group_keys=False
).apply(
    lambda x: x.sample(
        min(len(x), 10000),
        random_state=42
    ),
    include_groups=False
).reset_index(drop=True)

show_chart(
    px.histogram(
        sample,
        x="Amount",
        color="Transaction_Type",
        nbins=60,
        barmode="overlay",
        log_y=True
    ),
    "5. Transaction Amount Distribution (Sample)"
)

# CHART 6 - FRAUD RATE BY AMOUNT BAND
d = (
    df.groupby(
        "Amount_Band",
        observed=True
    )["Class"]
    .agg(
        Fraud_Rate="mean",
        Transactions="count"
    )
    .reset_index()
)

d["Fraud_Rate_Percent"] = (
    d["Fraud_Rate"] * 100
)

show_chart(
    px.bar(
        d,
        x="Amount_Band",
        y="Fraud_Rate_Percent",
        color="Transactions",
        hover_data=["Transactions"]
    ),
    "6. Fraud Rate by Transaction Amount Band"
)

# CHART 7 - ELAPSED HOUR
if "Elapsed_Hour" in df.columns:
    d = (
        df.groupby(
            "Elapsed_Hour", as_index=False
        )["Class"]
        .agg(
            Fraud_Count="sum",
            Total_Count="count"
        )
    )

    show_chart(
        px.line(
            d,
            x="Elapsed_Hour",
            y="Fraud_Count",
            markers=True
        ),
        "7. Fraud Cases by Elapsed Hour"
    )

# CHART 8 - PCA FEATURE COMPARISON
if {"V1", "V2"}.issubset(df.columns):
    fraud_sample = fraud.sample(
        min(len(fraud), 1000),
        random_state=42
    )

    legit_sample = legit.sample(
        min(len(legit), 5000),
        random_state=42
    )

    d = pd.concat(
        [fraud_sample, legit_sample],
        ignore_index=True
    )

    show_chart(
        px.scatter(
            d,
            x="V1",
            y="V2",
            color="Transaction_Type",
            opacity=0.55
        ),
        "8. Anonymized PCA Features V1 vs V2"
    )

# 6. CSV REPORTS
for group in [
    "Transaction_Type",
    "Amount_Band"
]:
    report = (
        df.groupby(
            group, observed=True
        )
        .agg(
            Transactions=("Class", "count"),
            Fraud_Cases=("Class", "sum"),
            Total_Amount=("Amount", "sum"),
            Average_Amount=("Amount", "mean")
        )
        .reset_index()
    )

    report["Fraud_Rate_Percent"] = (
        report["Fraud_Cases"]
        / report["Transactions"] * 100
    )

    report.to_csv(
        PROJECT / f"summary_{group.lower()}.csv",
        index=False
    )

if "Elapsed_Hour" in df.columns:
    report = (
        df.groupby("Elapsed_Hour")
        .agg(
            Transactions=("Class", "count"),
            Fraud_Cases=("Class", "sum")
        )
        .reset_index()
    )

    report.to_csv(
        PROJECT / "summary_elapsed_hour.csv",
        index=False
    )

# 7. COMBINED HTML DASHBOARD
html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Financial Fraud Intelligence</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    margin: 25px;
}
h1, p { text-align: center; }
.chart {
    background: white;
    border-radius: 12px;
    padding: 15px;
    margin: 20px auto;
    max-width: 1300px;
}
</style>
</head>
<body>
<h1>FINANCIAL FRAUD INTELLIGENCE</h1>
<p>Real Anonymized Credit Card Transaction Data</p>
"""

for i, (title, fig) in enumerate(figures):
    html += (
        '<div class="chart">'
        + to_html(
            fig,
            full_html=False,
            include_plotlyjs=(i == 0)
        )
        + "</div>"
    )

html += "</body></html>"

dashboard_path = (
    PROJECT / "Financial_Fraud_Dashboard.html"
)

dashboard_path.write_text(
    html, encoding="utf-8"
)

# 8. README
readme = f"""
PROJECT 3 - FINANCIAL FRAUD INTELLIGENCE

Original dataset:
https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Loaded from:
{source_used}

Records: {len(df)}
Fraud records: {len(fraud)}
Charts: {len(figures)}

No synthetic transactions generated.

Notes:
Class 1 = fraud, Class 0 = legitimate.
V1-V28 are anonymized PCA features.
Time, when available, is elapsed seconds,
not a real calendar timestamp.
Amount currency is not assumed.
The dataset covers approximately two days.
Charts are descriptive, not a fraud
prediction model.
"""

(PROJECT / "README.txt").write_text(
    readme, encoding="utf-8"
)

# 9. ZIP
zip_path = Path(
    "/content/Financial_Fraud_Intelligence.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:
    for file in PROJECT.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(
                    PROJECT.parent
                )
            )

# 10. FINAL OUTPUT
print("\n" + "=" * 45)
print("PROJECT 3 COMPLETED")
print("=" * 45)
print("Clean records:", len(df))
print("Fraud cases:", len(fraud))
print("Interactive charts:", len(figures))
print("Dashboard:", dashboard_path)
print("ZIP:", zip_path)
print("Automatic download: OFF")

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
